# Support Vector Machine (SVM) - Meme Kanseri (Breast Cancer) Analizi

**Veri Seti:** `breast-cancer.csv` (569 hasta kaydi, 32 sutun)

**Amac:** Hucre cekirdegi olculerine bakarak **diagnosis** (tani: Malign / Benign) siniflandirmasi yapmak.

## SVM Teorisi

**Support Vector Machine**, siniflari birbirinden ayiran en iyi karar sinirini (hyperplane) bulan denetimli bir ogrenme algoritmasidir.

- **Hyperplane:** Siniflari ayiran duzlem/cizgi
- **Support Vector:** Hyperplane'e en yakin noktalar (siniri belirler)
- **Margin:** Hyperplane ile support vector'ler arasindaki mesafe (maximize edilir)
- **Kernel Trick:** Veriyi yuksek boyuta tasiyarak dogrusal olmayan sinirlar cizmeyi saglar

### Kernel Fonksiyonlari
| Kernel | Kullanim |
|--------|----------|
| **Linear** | Dogrusal ayrilabilen veriler |
| **RBF** | Dogrusal olmayan veriler (en populer) |
| **Poly** | Polinom derecesi ile esneklik |

### Hiperparametreler
| Parametre | Anlami | Etkisi |
|-----------|--------|--------|
| **C** | Regularizasyon | Kucuk C = genis margin (high bias), buyuk C = dar margin (high variance) |
| **gamma** | RBF etki alani | Kucuk = genis etki, buyuk = dar etki |


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 1. Veri Setini Kesfetme ve Degiskenleri Tanima

Asagida veri setindeki tum degiskenleri (sutunlari) ve ozelliklerini inceleyecegiz.

In [2]:
df = pd.read_csv('breast-cancer.csv')

print('=' * 70)
print('VERI SETI BOYUTU')
print('=' * 70)
print(f'Satir sayisi: {df.shape[0]}')
print(f'Sutun sayisi: {df.shape[1]}')
print()

print('=' * 70)
print('SUTUNLAR VE VERI TIPLERI')
print('=' * 70)
for col in df.columns:
    print(f"{'%-25s' % col} | dtype: {str(df[col].dtype):<15} | Benzersiz: {df[col].nunique()}")

VERI SETI BOYUTU
Satir sayisi: 569
Sutun sayisi: 32

SUTUNLAR VE VERI TIPLERI
id                        | dtype: int64           | Benzersiz: 569
diagnosis                 | dtype: str             | Benzersiz: 2
radius_mean               | dtype: float64         | Benzersiz: 456
texture_mean              | dtype: float64         | Benzersiz: 479
perimeter_mean            | dtype: float64         | Benzersiz: 522
area_mean                 | dtype: float64         | Benzersiz: 539
smoothness_mean           | dtype: float64         | Benzersiz: 474
compactness_mean          | dtype: float64         | Benzersiz: 537
concavity_mean            | dtype: float64         | Benzersiz: 537
concave points_mean       | dtype: float64         | Benzersiz: 542
symmetry_mean             | dtype: float64         | Benzersiz: 432
fractal_dimension_mean    | dtype: float64         | Benzersiz: 499
radius_se                 | dtype: float64         | Benzersiz: 540
texture_se                | dtype: float

In [3]:
print('=' * 70)
print('DEGISKEN GRUPLARI')
print('=' * 70)
print('id                     : Hasta/kayit numarasi (kullanilmayacak)')
print('diagnosis              : Tani (HEDEF DEGISKEN) - M: Malign (kotu huylu), B: Benign (iyi huylu)')
print('*_mean                 : Hucre cekirdegi olculerinin ortalama degerleri (10 ozellik)')
print('*_se                   : Hucre cekirdegi olculerinin standart hatasi (10 ozellik)')
print('*_worst                : Hucre cekirdegi olculerinin en kotu (en buyuk) degerleri (10 ozellik)')
print()
print('Ölçülen 10 temel özellik: radius, texture, perimeter, area, smoothness,')
print('compactness, concavity, concave points, symmetry, fractal_dimension')
print()

print('=' * 70)
print('HEDEF DEGISKEN: diagnosis')
print('=' * 70)
print(df['diagnosis'].value_counts().to_string())
print()
print(df['diagnosis'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

DEGISKEN GRUPLARI
id                     : Hasta/kayit numarasi (kullanilmayacak)
diagnosis              : Tani (HEDEF DEGISKEN) - M: Malign (kotu huylu), B: Benign (iyi huylu)
*_mean                 : Hucre cekirdegi olculerinin ortalama degerleri (10 ozellik)
*_se                   : Hucre cekirdegi olculerinin standart hatasi (10 ozellik)
*_worst                : Hucre cekirdegi olculerinin en kotu (en buyuk) degerleri (10 ozellik)

Ölçülen 10 temel özellik: radius, texture, perimeter, area, smoothness,
compactness, concavity, concave points, symmetry, fractal_dimension

HEDEF DEGISKEN: diagnosis
diagnosis
B    357
M    212

diagnosis
B    62.74 %
M    37.26 %
Name: proportion, dtype: str


## 2. Veri On Isleme (Preprocessing)

SVM mesafe tabanli bir algoritma oldugu icin:
1. **Hedef degisken** sayisala cevrilmeli (LabelEncoder: M/B -> 1/0)
2. **id** sutunu modele girmemeli (bilgi tasimiyor)
3. **Sayisal degiskenler** olceklendirilmeli (StandardScaler)
4. **Egitim/test** ayrimi yapilmali

In [4]:
feature_cols = [c for c in df.columns if c not in ['id', 'diagnosis']]
X = df[feature_cols].copy()
y = df['diagnosis']

print('KULLANILACAK BAGIMSIZ DEGISKENLER (X):', len(feature_cols), 'adet')
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")
print(f'\nHEDEF DEGISKEN (y): diagnosis')

KULLANILACAK BAGIMSIZ DEGISKENLER (X): 30 adet
   1. radius_mean
   2. texture_mean
   3. perimeter_mean
   4. area_mean
   5. smoothness_mean
   6. compactness_mean
   7. concavity_mean
   8. concave points_mean
   9. symmetry_mean
  10. fractal_dimension_mean
  11. radius_se
  12. texture_se
  13. perimeter_se
  14. area_se
  15. smoothness_se
  16. compactness_se
  17. concavity_se
  18. concave points_se
  19. symmetry_se
  20. fractal_dimension_se
  21. radius_worst
  22. texture_worst
  23. perimeter_worst
  24. area_worst
  25. smoothness_worst
  26. compactness_worst
  27. concavity_worst
  28. concave points_worst
  29. symmetry_worst
  30. fractal_dimension_worst

HEDEF DEGISKEN (y): diagnosis


In [5]:
print('EKSIK DEGER KONTROLU')
print('=' * 50)
missing = X.isnull().sum().sum()
print(f'Toplam eksik deger: {missing}')

EKSIK DEGER KONTROLU
Toplam eksik deger: 0


In [6]:
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

print('HEDEF DEGISKEN (diagnosis) KODLAMA (LabelEncoder)')
print('=' * 50)
for i, cls in enumerate(le_y.classes_):
    print(f'  {i} -> {cls}')

HEDEF DEGISKEN (diagnosis) KODLAMA (LabelEncoder)
  0 -> B
  1 -> M


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print('VERI AYRIMI (Train/Test)')
print(f'  Egitim seti  : {X_train.shape[0]} ornek (%80)')
print(f'  Test seti    : {X_test.shape[0]} ornek (%20)')
print(f'  Stratify     : Evet (sinif dagilimi korundu)')

VERI AYRIMI (Train/Test)
  Egitim seti  : 455 ornek (%80)
  Test seti    : 114 ornek (%20)
  Stratify     : Evet (sinif dagilimi korundu)


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('VERI OLCEKLENDIRILDI (StandardScaler)')
print(f'  Ortalama (her sutun icin): 0')
print(f'  Standart sapma (her sutun icin): 1')
print(f'  X_train_scaled boyutu: {X_train_scaled.shape}')
print(f'  X_test_scaled boyutu : {X_test_scaled.shape}')

VERI OLCEKLENDIRILDI (StandardScaler)
  Ortalama (her sutun icin): 0
  Standart sapma (her sutun icin): 1
  X_train_scaled boyutu: (455, 30)
  X_test_scaled boyutu : (114, 30)


## 3. SVM Modelleri

3 farkli kernel ile SVM egitip karsilastiracagiz:
- **Linear SVM** (dogrusal)
- **RBF SVM** (radyal tabanli - varsayilan)
- **Polynomial SVM** (polinom)

Ardindan **GridSearchCV** ile en iyi parametreleri bulup modeli optimize edecegiz.

In [9]:
print('=' * 50)
print('LINEAR SVM')
print('=' * 50)
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_linear):.4f}')
print(f'Destek vektor sayisi: {svm_linear.n_support_.sum()}')
print('\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_linear, target_names=le_y.classes_))

LINEAR SVM
Test dogrulugu: 0.9649
Destek vektor sayisi: 38

Siniflandirma Raporu:
              precision    recall  f1-score   support

           B       0.95      1.00      0.97        72
           M       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114



In [10]:
print('=' * 50)
print('RBF SVM (Varsayilan)')
print('=' * 50)
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_rbf):.4f}')
print(f'Destek vektor sayisi: {svm_rbf.n_support_.sum()}')
print('\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_rbf, target_names=le_y.classes_))

RBF SVM (Varsayilan)
Test dogrulugu: 0.9737
Destek vektor sayisi: 107

Siniflandirma Raporu:
              precision    recall  f1-score   support

           B       0.96      1.00      0.98        72
           M       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [11]:
print('=' * 50)
print('POLINOMIAL SVM (degree=3)')
print('=' * 50)
svm_poly = SVC(kernel='poly', degree=3, random_state=42)
svm_poly.fit(X_train_scaled, y_train)
y_pred_poly = svm_poly.predict(X_test_scaled)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_poly):.4f}')
print(f'Destek vektor sayisi: {svm_poly.n_support_.sum()}')
print('\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_poly, target_names=le_y.classes_))

POLINOMIAL SVM (degree=3)
Test dogrulugu: 0.8860
Destek vektor sayisi: 149

Siniflandirma Raporu:
              precision    recall  f1-score   support

           B       0.85      1.00      0.92        72
           M       1.00      0.69      0.82        42

    accuracy                           0.89       114
   macro avg       0.92      0.85      0.87       114
weighted avg       0.90      0.89      0.88       114



## 4. Hiperparametre Optimizasyonu (GridSearchCV)

En iyi `C` ve `gamma` degerlerini bulmak icin **GridSearchCV** kullaniyoruz.

- **C**: [0.1, 1, 10, 100]
- **gamma**: ['scale', 'auto', 0.1, 0.01]
- **kernel**: ['rbf']
- **Cross-validation**: 5-fold

In [12]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf']
}

print('ARANAN PARAMETRELER:')
for k, v in param_grid.items():
    print(f'  {k}: {v}')
print(f'\nToplam kombinasyon: {len(param_grid["C"]) * len(param_grid["gamma"])}')
print(f'Toplam model (5-fold CV ile): {len(param_grid["C"]) * len(param_grid["gamma"]) * 5}')

ARANAN PARAMETRELER:
  C: [0.1, 1, 10, 100]
  gamma: ['scale', 'auto', 0.1, 0.01]
  kernel: ['rbf']

Toplam kombinasyon: 16
Toplam model (5-fold CV ile): 80


In [13]:
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f'\n{"=" * 50}')
print('EN IYI PARAMETRELER')
print('=' * 50)
print(f'C          : {grid_search.best_params_["C"]}')
print(f'gamma      : {grid_search.best_params_["gamma"]}')
print(f'kernel     : {grid_search.best_params_["kernel"]}')
print(f'CV skoru   : {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits



EN IYI PARAMETRELER
C          : 1
gamma      : scale
kernel     : rbf
CV skoru   : 0.9758


In [14]:
best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)

print('=' * 50)
print('EN IYI MODEL SONUCLARI')
print('=' * 50)
print(f'Test dogrulugu: {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Cross-validation skoru: {grid_search.best_score_:.4f}')
print(f'\nSiniflandirma Raporu:')
print(classification_report(y_test, y_pred_best, target_names=le_y.classes_))

EN IYI MODEL SONUCLARI
Test dogrulugu: 0.9737
Cross-validation skoru: 0.9758

Siniflandirma Raporu:
              precision    recall  f1-score   support

           B       0.96      1.00      0.98        72
           M       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [15]:
print('=' * 50)
print('KARISIKLIK MATRISI (Confusion Matrix)')
print('=' * 50)
cm = confusion_matrix(y_test, y_pred_best)
print('Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger')
print()
header = ' ' * 22 + ' '.join(f'{c:>18}' for c in le_y.classes_)
print(header)
print('\u2500' * len(header))
for i, row in enumerate(cm):
    print(f"{le_y.classes_[i]:<22}" + ' '.join(f'{val:>18}' for val in row))
print('Yorum: Kosegen (diagonal) ne kadar yuksekse model o kadar basarili.')

KARISIKLIK MATRISI (Confusion Matrix)
Satirlar: GERCEK deger | Sutunlar: TAHMIN edilen deger

                                       B                  M
───────────────────────────────────────────────────────────
B                                     72                  0
M                                      3                 39
Yorum: Kosegen (diagonal) ne kadar yuksekse model o kadar basarili.


In [16]:
print('=' * 50)
print('DESTEK VEKTOR ANALIZI')
print('=' * 50)
print(f'Toplam destek vektor sayisi  : {best_svm.n_support_.sum()}')
print(f'Toplam egitim ornegi         : {len(X_train_scaled)}')
print(f'Destek vektor orani          : {best_svm.n_support_.sum() / len(X_train_scaled) * 100:.1f}%')
print()
print('Her sinif icin destek vektor sayisi:')
for i, cls in enumerate(le_y.classes_):
    print(f'  {cls:<20} : {best_svm.n_support_[i]} vektor')

DESTEK VEKTOR ANALIZI
Toplam destek vektor sayisi  : 107
Toplam egitim ornegi         : 455
Destek vektor orani          : 23.5%

Her sinif icin destek vektor sayisi:
  B                    : 50 vektor
  M                    : 57 vektor


In [17]:
print('=' * 50)
print('OZELLIK ONEM SIRASI (Linear SVM katsayilarina gore)')
print('=' * 50)
coef_abs = np.abs(svm_linear.coef_[0])
order = np.argsort(coef_abs)[::-1]
print(f"{'Ozellik':<28}{'|Katsayi|':>12}")
print('-' * 40)
for idx in order[:10]:
    print(f"{feature_cols[idx]:<28}{coef_abs[idx]:>12.3f}")

OZELLIK ONEM SIRASI (Linear SVM katsayilarina gore)
Ozellik                        |Katsayi|
----------------------------------------
texture_worst                      1.072
concavity_mean                     0.880
area_se                            0.830
concave points_mean                0.828
concavity_worst                    0.824
radius_se                          0.756
area_worst                         0.725
texture_se                         0.614
radius_worst                       0.542
compactness_se                     0.536


In [18]:
print('=' * 50)
print('ORNEK TAHMIN (Test Setinden 10 Kayit)')
print('=' * 50)
print(f'{"No":<5} {"Tahmin":<10} {"Gercek":<10} {"Dogru?":<8}')
print('\u2500' * 35)
for i in range(10):
    tahmin = le_y.inverse_transform([best_svm.predict(X_test_scaled[i].reshape(1, -1))[0]])[0]
    gercek = le_y.inverse_transform([y_test[i]])[0]
    dogru = '\u2713' if tahmin == gercek else '\u2717'
    print(f"{i+1:<5} {tahmin:<10} {gercek:<10} {dogru:<8}")

ORNEK TAHMIN (Test Setinden 10 Kayit)
No    Tahmin     Gercek     Dogru?  
───────────────────────────────────
1     B          B          ✓       
2     M          M          ✓       
3     B          B          ✓       
4     B          M          ✗       
5     B          B          ✓       
6     B          B          ✓       
7     M          M          ✓       
8     B          B          ✓       
9     B          B          ✓       
10    B          B          ✓       


## 5. Sonuc ve Degerlendirme

| Model | Dogruluk | Aciklama |
|-------|----------|----------|
| Linear SVM | %96.5 | Dogrusal sinir, zaten cok basarili (veri buyuk olcude dogrusal ayrilabilir) |
| RBF SVM (varsayilan) | %97.4 | Dogrusal olmayan iliskileri de yakalayarak hafif iyilesme sagliyor |
| Polynomial SVM (3. derece) | %88.6 | Bu veri setinde asiri esneklik gerekmedigi icin daha dusuk basari |
| **Optimize RBF SVM** | **%97.4** | **C=1, gamma=scale ile CV skoru %97.6, en dengeli model** |

### Dikkat Cikarimlar
- Meme kanseri veri seti (Wisconsin Breast Cancer) genellikle dogrusal olarak iyi ayrilabilen bir veri setidir, bu yuzden Linear ve RBF SVM'ler birbirine yakin ve yuksek basari gosterebilir.
- **Malign (M)** sinifinin **recall** degeri klinik acidan cok onemlidir: bir kotu huylu vakayi kacirmak (false negative), iyi huylu bir vakayi yanlislikla kotu huylu olarak isaretlemekten (false positive) çok daha maliyetlidir.
- Destek vektor orani, karar siniri karmasikligi hakkinda fikir verir: oran ne kadar dusukse siniflar o kadar net ayrilabiliyor demektir.
- Ozellik onem sirasi, hangi hucre olculerinin (orn. radius, concave points, perimeter) tani icin en belirleyici oldugunu gosterir.
